<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_45_websocket_chat/note_lesson_45_chat_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💬 Урок 45 — WebSockets + практика: чат

| Крок | Що робимо |
|---|---|
| 0 | `crispy_notes_project` уроку 45 у ноутбуці: Олена, Анна, Боб і група «Сім'я» |
| 1 | підключення: учасник і чужий (вправа 1) |
| 2 | розсилка і історія (вправи 2–3) |
| 3 | правила в `services`, а не в consumer (вправа 4) |
| 4 | channel layer зсередини: групи і `type` → метод (вправи 5–6) |
| 5 | права під час з'єднання і `Origin` (вправи 7–8) |
| 6 | XSS: хто екранує текст (вправа 9) |
| 7 | знайди помилку: перевірка лише в `connect()` (вправа 10) |

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам. Сервер і браузер не потрібні: `WebsocketCommunicator` — це «браузер» з тестів, він говорить з consumer напряму, а channel layer живе в пам'яті процесу. Зміни коду — у книзі курсу: [Урок 45](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_45/); теорія — крок 7B Django-книги [Async Django і WebSocket](https://nikoriakviktot.github.io/notes_chat_app/tutorials/07_async_django/).

---
## 0. Проєкт у ноутбуці

У Colab клітинка завантажить папку уроку 45, встановить залежності (`channels`, `daphne`, `channels-redis` і все з уроку 44) і створить чисту базу SQLite. Redis не потрібен: без `REDIS_URL` проєкт бере `InMemoryChannelLayer`.

In [ ]:
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_45_websocket_chat"
REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("crispy_notes_project"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

try:
    import channels, crispy_bootstrap5, daphne, debug_toolbar, drf_spectacular, rest_framework_simplejwt  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "crispy_notes_project/requirements.txt"], check=True)

os.chdir("crispy_notes_project")
for name in ("DATABASE_URL", "REDIS_URL"):
    os.environ.pop(name, None)                          # SQLite і channel layer у пам'яті
if os.path.exists("db.sqlite3"):
    os.remove("db.sqlite3")                             # щоразу — чиста база
result = subprocess.run([sys.executable, "manage.py", "migrate"], capture_output=True, text=True)
print("\n".join((result.stdout or result.stderr).splitlines()[-2:]))

In [ ]:
import json

import django

sys.path.insert(0, os.getcwd())
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hello_project.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"      # Jupyter працює в asyncio-циклі (урок 33)
django.setup()

from channels.db import database_sync_to_async
from channels.layers import get_channel_layer
from channels.testing import WebsocketCommunicator
from django.conf import settings
from django.contrib.auth.models import User
from django.core.exceptions import PermissionDenied, ValidationError

from hello_app import selectors, services
from hello_app.consumers import CLOSE_NOT_ALLOWED, GroupChatConsumer
from hello_app.models import ChatMessage

olena = User.objects.create_user("olena", password="pass-12345")
ann = User.objects.create_user("ann", password="pass-12345")
bob = User.objects.create_user("bob", password="pass-12345")
family = services.create_group(name="Сім'я", creator=olena)
services.add_user_to_group(family, "ann")


def communicator_for(group, user, consumer=GroupChatConsumer):
    """«Браузер» користувача user на сторінці чату групи group — як у tests_chat.py."""
    communicator = WebsocketCommunicator(consumer.as_asgi(), f"/ws/groups/{group.pk}/chat/")
    communicator.scope["user"] = user                  # замість AuthMiddlewareStack
    communicator.scope["url_route"] = {"kwargs": {"group_pk": str(group.pk)}}  # замість URLRouter
    return communicator


print("channel layer:", settings.CHANNEL_LAYERS["default"]["BACKEND"])
print("група:", family.name, [u.username for u in family.user_set.order_by("username")])

---
## 1. Підключення

Шлях одного повідомлення від Анни до Олени — покроково (консюмер Анни і консюмер Олени — два окремі об'єкти, їх з'єднує channel layer):

```mermaid
flowchart TD
    classDef step     fill:#263238,stroke:#90a4ae,color:#ffffff;
    classDef decision fill:#37474f,stroke:#64b5f6,color:#ffffff;
    classDef success  fill:#1b5e20,stroke:#4CAF50,color:#ffffff;
    classDef error    fill:#4e1f1f,stroke:#f44336,color:#ffffff;
    classDef warning  fill:#4a3b00,stroke:#ff9800,color:#ffffff;

    subgraph C1["connect(): Анна відкриває сторінку чату"]
        direction LR
        a1["is_group_member(ann, family)"] --> a2["так → group_add<br>chat_group_1"] --> a3["accept()<br>+ history"]
    end
    subgraph C2["receive(): Анна пише «Привіт!»"]
        direction LR
        b1["services.post_chat_message"] --> b2["ChatMessage<br>збережено"] --> b3["group_send<br>type: chat_message"]
    end
    subgraph C3["channel layer: група chat_group_1"]
        direction LR
        c1["канал Олени"] ~~~ c2["канал Анни"]
    end
    subgraph C4["chat_message(): кожен consumer групи"]
        direction LR
        d1["Олена отримує<br>type: message"] ~~~ d2["Анна отримує<br>своє ж"]
    end
    C1 --> C2 --> C3 --> C4

    class a1 decision
    class a2,b1,c1,c2 step
    class b2,b3 warning
    class a3,d1,d2 success
```

### Вправа 1

Підключи Олену й Боба до чату «Сім'ї» (`await communicator.connect()` повертає пару `(підключено, код)`). Збережи пари в `olena_result` і `bob_result`.

**🔮 Прогноз:** що отримає Боб — з'єднання, яке одразу закриється, чи відмову ще в рукостисканні?

In [ ]:
# YOUR CODE HERE

print("Олена:", olena_result, "| Боб:", bob_result)
assert olena_result[0] is True and bob_result[0] is False
print("✅ Вправа 1 пройдена")

<details>
<summary>Відповідь</summary>

Боб отримує `(False, 1000)` — відмову в рукостисканні: consumer викликає `close()` **до** `accept()`, тож з'єднання так і не відкривається. Браузер у такому разі бачить код `1006`, як при обриві мережі. Перевірка — `selectors.is_group_member`, та сама, що для сторінок групи в уроці 44. Consumer не пише свого правила.
</details>

---
## 2. Розсилка й історія

### Вправа 2

Підключи Анну (`ann_ws`). Олена надсилає `{"content": "  Привіт!  "}` (`send_json_to`). Отримай кадр у Олени й у Анни (`receive_json_from`) — `frame_olena`, `frame_ann` — і порахуй `saved` — скільки `ChatMessage` у базі.

In [ ]:
# YOUR CODE HERE

print("Олена:", frame_olena)
print("Анна: ", frame_ann)
assert frame_olena == frame_ann
assert (frame_ann["type"], frame_ann["author"], frame_ann["content"]) == ("message", "olena", "Привіт!")
assert saved == 1
print("✅ Вправа 2 пройдена")

**🔮 Прогноз:** звідки взялися пробіли, яких немає в кадрі? Чому Олена отримала **своє** повідомлення?

<details>
<summary>Відповідь</summary>

Пробіли прибрав `services.post_chat_message` (`strip()`), тож правило тексту одне для будь-якого транспорту. Олена отримала своє повідомлення, бо `group_send` іде **всій** групі, разом з автором. Так браузер показує повідомлення лише тоді, коли сервер його прийняв і зберіг.
</details>

### Вправа 3

Анна дописує ще два повідомлення (`«Що купити?»`, `«Хліб»`). Підключи **другу вкладку** Олени (`olena_tab2`) і збери `content` усіх кадрів історії (`type == "history"`) у список `history` — по порядку, як вони прийшли.

In [ ]:
# YOUR CODE HERE

print("історія:", history)
assert history == ["Привіт!", "Що купити?", "Хліб"]
print("✅ Вправа 3 пройдена — від старих до нових")

<details>
<summary>Відповідь</summary>

`selectors.recent_chat_messages` бере 50 **найновіших** (`order_by('-timestamp', '-id')[:50]`) і перевертає їх — від старих до нових, як читають чат. Повертає **список**, а не QuerySet: consumer викликає його через `database_sync_to_async`, і SQL має виконатися в потоці, а не в циклі подій.
</details>

---
## 3. Правила — у services

### Вправа 4

Без WebSocket: виклич `services.post_chat_message(group_pk=family.pk, author=…, content=…)` для трьох випадків і збери назви винятків у `errors`:

1. Олена, `"   "`;
2. Олена, `"я" * 2001`;
3. Боб, `"Привіт"`.

In [ ]:
cases = [(olena, "   "), (olena, "я" * 2001), (bob, "Привіт")]

# YOUR CODE HERE

print(errors)
assert errors == ["ValidationError", "ValidationError", "PermissionDenied"]
print("✅ Вправа 4 пройдена")

**🔮 Прогноз:** що зробить consumer з кожним із цих винятків?

<details>
<summary>Відповідь</summary>

`ValidationError` — проігнорує повідомлення (як і старий код з порожнім текстом). `PermissionDenied` — закриє з'єднання кодом `4403`: людина вже не учасник. Правило живе в сервісі, а consumer лише перекладає відповідь сервісу на мову WebSocket.
</details>

---
## 4. Channel layer зсередини

`group_send` не знає про consumers. Він кладе **словник** у канал кожного учасника групи, а consumer, який слухає канал, викликає метод з ім'ям `type` (крапки → підкреслення).

### Вправа 5

Візьми `layer = get_channel_layer()`, створи канал (`await layer.new_channel()`), додай його в групу `"demo"`, надішли в групу `{"type": "member.removed", "user_id": 42}` і прочитай подію з каналу (`await layer.receive(channel)`) — `event`. Знайди ім'я методу, який її обробить, — `handler` — і перевір, що такий метод у `GroupChatConsumer` є.

In [ ]:
# YOUR CODE HERE

print("подія:", event, "→ метод:", handler)
assert event == {"type": "member.removed", "user_id": 42}
assert handler == "member_removed" and hasattr(GroupChatConsumer, handler)
print("✅ Вправа 5 пройдена")

### Вправа 6. Групи не змішуються

Олена створює групу «Робота» з Бобом. Боб підключається до чату «Роботи» (`bob_work`). Олена пише в «Сім'ю» `«Тільки для сім'ї»`. Перевір: Анна отримала повідомлення (`ann_got`), а Боб — нічого (`bob_silent = await bob_work.receive_nothing(timeout=0.3)`).

In [ ]:
# YOUR CODE HERE

print("Анна:", ann_got, "| Боб нічого не отримав:", bob_silent)
assert ann_got == "Тільки для сім'ї" and bob_silent
print("✅ Вправа 6 пройдена — одна група channel layer на групу нотаток:", services.chat_group_name(family.pk))

---
## 5. Права — протягом усього з'єднання

WebSocket живе годинами. Перевірка в `connect()` — це лише «на вході».

### Вправа 7

Олена вилучає Анну з групи — `services.remove_user_from_group(family, ann)`. Виклич його через `database_sync_to_async` (він синхронний і сам надсилає подію в channel layer). Збережи, що отримала вкладка Анни, в `ann_output` (`await ann_ws.receive_output(timeout=1)`), а що Олена — в `olena_silent` (`receive_nothing`).

In [ ]:
# YOUR CODE HERE

print("Анна:", ann_output, "| Олені нічого:", olena_silent)
assert ann_output == {"type": "websocket.close", "code": CLOSE_NOT_ALLOWED}
assert olena_silent
print("✅ Вправа 7 пройдена — код", CLOSE_NOT_ALLOWED)

<details>
<summary>Відповідь</summary>

`remove_user_from_group` після COMMIT надсилає в групу подію `member.removed` з `user_id` Анни. Її отримують **усі** consumers групи, але закривається лише той, чий `self.user.pk` збігається. Код `4403` — з діапазону застосунку (4000–4999): JS-клієнт на нього не перепідключається і показує «Немає доступу до чату цієї групи».
</details>

### Вправа 8. `Origin`

Cookie сесії браузер надсилає на наш сайт з **будь-якої** сторінки. Збери стек з `asgi.py` (`websocket_application()`) з `ALLOWED_HOSTS=["localhost"]` і підключися з cookie Олени — зі своєї сторінки `http://localhost:8000` (`own`) і з `https://evil.example` (`evil`).

In [ ]:
from django.test import Client, override_settings

from hello_project.asgi import websocket_application

browser = Client()
browser.force_login(olena)
cookie = f"sessionid={browser.cookies['sessionid'].value}".encode()


async def connect_from(origin):
    communicator = WebsocketCommunicator(
        websocket_application(), f"/ws/groups/{family.pk}/chat/",
        headers=[(b"host", b"localhost"), (b"cookie", cookie), (b"origin", origin)],
    )
    connected, _ = await communicator.connect()
    await communicator.disconnect()
    return connected

# YOUR CODE HERE

print("своя сторінка:", own, "| чужий сайт:", evil)
assert own is True and evil is False
print("✅ Вправа 8 пройдена")

**🔮 Прогноз:** чому `websocket_application()` — функція, а не просто змінна з готовим стеком?

<details>
<summary>Відповідь</summary>

`AllowedHostsOriginValidator` читає `ALLOWED_HOSTS` (і `DEBUG`) **один раз**, коли його створюють. Стек, зібраний при імпорті `asgi.py`, не побачив би `override_settings`. Тому тести й ця вправа збирають стек заново. На сервері це означає: без `DJANGO_ALLOWED_HOSTS` чат не підключиться з жодної сторінки.
</details>

---
## 6. XSS: хто екранує

### Вправа 9

1. Олена надсилає `payload` — отримай кадр і збережи `content` у `echoed`.
2. Напиши `escape_html(text)` — ту саму заміну, що `escapeHtml` у `group_chat.js` (`&`, `<`, `>`, `"`, `'` → `&amp;`, `&lt;`, `&gt;`, `&quot;`, `&#039;`, **саме в такому порядку**).

In [ ]:
payload = '<img src=x onerror="alert(\'x\')">&'

# YOUR CODE HERE

print("сервер повернув:", echoed)
print("у браузері стане:", escape_html(echoed))
assert echoed == payload
assert escape_html(payload) == "&lt;img src=x onerror=&quot;alert(&#039;x&#039;)&quot;&gt;&amp;"
print("✅ Вправа 9 пройдена")

**🔮 Прогноз:** чому `&` замінюють **першим**?

<details>
<summary>Відповідь</summary>

Інакше `<` → `&lt;`, а потім `&` у ньому → `&amp;lt;`: у браузері замість `<` з'явиться `&lt;`. Сервер зберігає текст **як є** — він не знає, де його покажуть (HTML, JSON API, push-сповіщення). Екранує той, хто вставляє в HTML: JS-клієнт перед `innerHTML`. Тест `JsEscapeTests` запускає справжню функцію з `group_chat.js` у Node.js.
</details>

---
## 7. Знайди помилку

`OldConsumer` — consumer з перенесеного коду: членство перевіряється лише в `connect()`, повідомлення зберігаються напряму, подію вилучення він не обробляє. Його тест «чужого не пускає» зелений.

### Вправа 10

Анна знову в групі (клітинка нижче повертає її). Підключи Анну через `OldConsumer`, вилучи її з групи через `services`, а тоді Анна пише `«Ще тут?»`. Збережи:

- `still_member` — чи Анна в групі (`selectors.is_group_member`);
- `saved_after_removal` — скільки повідомлень Анни з текстом «Ще тут?» у базі.

In [ ]:
class OldConsumer(GroupChatConsumer):
    async def receive(self, text_data):
        content = json.loads(text_data).get("content", "").strip()
        if not content or len(content) > 2000:
            return
        msg = await database_sync_to_async(ChatMessage.objects.create)(
            group_id=self.group_pk, author=self.user, content=content)
        await self.channel_layer.group_send(self.room_group_name, {
            "type": "chat_message", "message_id": msg.id, "author": self.user.username,
            "content": msg.content, "timestamp": msg.timestamp.isoformat()})

    async def member_removed(self, event):
        pass                                           # старий код такої події не знав


services.add_user_to_group(family, "ann")

# YOUR CODE HERE

print("Анна в групі:", still_member, "| збережено від Анни після вилучення:", saved_after_removal)
assert still_member is False and saved_after_removal == 1
print("✅ Вправа 10 пройдена — помилку відтворено")

<details>
<summary>Відповідь</summary>

Членство перевірено **один раз**. Хто був учасником при підключенні, той лишається ним для цього consumer: його канал досі в групі channel layer (отримує нові повідомлення), а `receive` не перевіряє нічого. Тест дивився лише на **момент підключення**. У HTTP кожен запит — новий «connect», тож цього досить. У WebSocket права перевіряють на кожне повідомлення (`services.post_chat_message` → `PermissionDenied` → `close(4403)`) і за подією зміни прав (`member.removed`), як у вправі 7.
</details>

In [ ]:
for communicator in (olena_ws, ann_ws):
    await communicator.disconnect()
print("повідомлень у чаті «Сім'ї»:", len(selectors.recent_chat_messages(family.pk)))

---
## Самоперевірка

1. Чим WebSocket відрізняється від HTTP-запиту і чому для чату не досить опитувати сервер?
2. Навіщо `database_sync_to_async` і чому selector повертає список, а не QuerySet?
3. Що кладе `group_send` у канал і як consumer вирішує, який метод викликати?
4. Назви всі місця, де перевіряються права в чаті, від рукостискання до вилучення з групи.
5. Коли `InMemoryChannelLayer` перестає працювати і що тоді дає Redis?

## Далі

- Книга курсу: [Урок 45](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_45/) — ASGI-стек, покрокові діаграми з'єднання й вилучення, архітектура до/після.
- Наживо: у папці `crispy_notes_project` — `python manage.py runserver` (daphne), дві вкладки різних користувачів на сторінці групи → «Чат». З Redis: `docker compose up -d redis` і `REDIS_URL=redis://localhost:6379/0`.
- Тести: `python manage.py test hello_app.tests_chat hello_app.tests_consumers`.